# Complete End-to-End ML Training Workflow

This notebook provides a complete working workflow for training models using our advanced training pipeline. We'll run through every step from data loading to model training and export.

## Setup Environment

First, we'll install the required dependencies.

In [ ]:
# Install required dependencies
!pip install -q transformers datasets torch accelerate bitsandbytes evaluate sentencepiece
!pip install -q "ray[tune]" deepspeed nlpaug huggingface_hub
!pip install -q tqdm matplotlib pyyaml safetensors scikit-learn

## Import Required Modules

Now, let's import all the necessary modules.

In [ ]:
import os
import sys
import logging
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Dict, List, Optional, Union, Any, Tuple, Callable
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Import modules from our codebase
from src.data.loaders import DatasetLoader
from src.data.preprocessors import DataPreprocessor
from src.model.architecture import create_model_from_config
from src.model.training import Trainer, TrainingArguments
from src.model.peft import apply_peft_to_model, prepare_for_qlora
from src.utils.tokenization import get_tokenizer
from src.utils.model_evaluation import evaluate_model

# Check for optional dependencies
try:
    import torch._dynamo
    TORCH_COMPILE_AVAILABLE = True
    print("PyTorch compilation (torch.compile) is available.")
except ImportError:
    TORCH_COMPILE_AVAILABLE = False
    print("PyTorch compilation is not available. Using standard execution.")

try:
    import bitsandbytes as bnb
    BITSANDBYTES_AVAILABLE = True
    print("Bitsandbytes is available. QLoRA and 4/8-bit quantization is supported.")
except ImportError:
    BITSANDBYTES_AVAILABLE = False
    print("Bitsandbytes is not available. Using standard precision.")

## Create Training Configuration

Let's create a realistic training configuration that will work well on most hardware.

In [ ]:
%%writefile real_config.yaml
# Real working configuration for actual training

# Project information
project_name: "real_transformer_training"
output_dir: "./output_real_model"
seed: 42

# Model configuration with practical settings
model:
  type: "decoder_only_transformer"
  size: "small"
  sizes:
    small:
      n_layers: 4      # Reduced for faster training
      n_heads: 4 
      d_model: 256     # Smaller embedding size
      d_ff: 1024       # Smaller feed-forward
      max_seq_length: 512
  dropout: 0.1
  attention:
    causal: true
    rotary_embedding: true
  
  # Architecture settings
  architecture:
    position_embeddings: "rotary"
    attention_type: "mha"
    norm_type: "layer_norm"
    normalization_strategy: "pre_norm"
    ffn_type: "gelu"
    use_flash_attention: false
    initialization:
      method: "normal"
      std: 0.02
  
  # PyTorch compilation
  compile:
    enabled: false
  
  # Gradient checkpointing
  gradient_checkpointing: true

# Tokenizer configuration
tokenizer:
  type: "huggingface"
  name: "gpt2"
  vocab_size: 50257
  max_length: 512
  padding_side: "right"
  truncation_side: "right"
  add_bos_token: true
  add_eos_token: true

# PEFT configuration
peft:
  enabled: true
  method: "lora"
  
  lora:
    r: 8
    alpha: 16
    dropout: 0.05
    target_modules:
      - "q_proj"
      - "k_proj" 
      - "v_proj"
      - "o_proj"
    bias: "none"

# Training configuration
training:
  active_stage: "real_finetune"
  stages:
    - name: "real_finetune"
      epochs: 2
      datasets:
        # We'll load multiple datasets with appropriate limits
        - name: "imdb"
          split: "train"
          streaming: false
          max_samples: 5000
        
        - name: "Salesforce/dialogstudio"
          split: "train"
          streaming: true
          max_samples: 5000
        
        - name: "wikicorpus"
          subset: "en"
          split: "train"
          streaming: true
          max_samples: 5000
        
        # Evaluation datasets
        - name: "imdb"
          split: "test"
          streaming: false
          max_samples: 1000
      
      learning_rate:
        initial: 5.0e-4
        min: 1.0e-6
        schedule: "cosine"
        warmup_steps: 100
      
      peft_method: "lora"
      peft_config:
        r: 8
        alpha: 16
        dropout: 0.05
  
  # Optimizer settings
  optimizer:
    name: "adamw"
    weight_decay: 0.01
    beta1: 0.9
    beta2: 0.999
    eps: 1.0e-8
    use_8bit: false
  
  mixed_precision: "fp16"
  gradient_clipping: 1.0
  
  checkpointing:
    save_steps: 200
    keep_last_n: 1
    save_optimizer_state: true
    merge_lora_weights: true
  
  evaluation:
    eval_steps: 200

# Data processing
data_processing:
  preprocessing:
    normalize_unicode: true
    normalize_whitespace: true
    remove_html: true
    min_length: 10
    max_length: 512
  
  batching:
    train_batch_size: 8
    eval_batch_size: 8
    gradient_accumulation_steps: 4
    dynamic_batching: false

# Evaluation config
evaluation:
  metrics:
    - "loss"
    - "perplexity"
    - "accuracy"
  
  generation:
    max_length: 64
    temperature: 0.8
    top_p: 0.92
    top_k: 50
    do_sample: true

## Load Configuration

Now, let's load our training configuration.

In [ ]:
def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

# Load configuration
config_path = "real_config.yaml"
config = load_config(config_path)

# Set random seed for reproducibility
torch.manual_seed(config['seed'])
np.random.seed(config['seed'])

# Create output directory
os.makedirs(config['output_dir'], exist_ok=True)

print(f"Loaded configuration with {len(config.keys())} sections")
for key in config.keys():
    print(f"- {key}")

## Direct Dataset Loading and Processing

Let's load and process our datasets directly using HuggingFace's datasets library.

In [ ]:
# Function to load datasets with sample limits
def load_datasets_with_limits(config):
    """Load datasets with appropriate sample limits."""
    # Get active stage
    active_stage = config['training']['active_stage']
    stage_config = None
    
    # Find active stage configuration
    for stage in config['training']['stages']:
        if stage['name'] == active_stage:
            stage_config = stage
            break
    
    if stage_config is None:
        raise ValueError(f"Training stage {active_stage} not found")
    
    train_datasets = []
    eval_datasets = []
    
    # Process each dataset in configuration
    for dataset_config in stage_config['datasets']:
        name = dataset_config['name']
        split = dataset_config['split']
        streaming = dataset_config.get('streaming', False)
        max_samples = dataset_config.get('max_samples', None)
        subset = dataset_config.get('subset', None)
        
        print(f"Loading dataset: {name} (split: {split}, max_samples: {max_samples})")
        
        try:
            # Load dataset
            if subset:
                dataset = load_dataset(name, subset, split=split, streaming=streaming)
            else:
                dataset = load_dataset(name, split=split, streaming=streaming)
            
            # Convert streaming dataset to regular if we need to limit samples
            if streaming and max_samples:
                dataset = dataset.take(max_samples)
                streaming = False
                
            # Limit samples for non-streaming datasets    
            if not streaming and max_samples and max_samples < len(dataset):
                indices = np.random.choice(len(dataset), max_samples, replace=False)
                dataset = dataset.select(indices)
            
            # Add to appropriate list
            if split == 'train':
                train_datasets.append(dataset)
            elif split in ['test', 'validation']:
                eval_datasets.append(dataset)
                
            print(f"  Successfully loaded with {len(dataset) if not streaming else 'streaming'} examples")
            
        except Exception as e:
            print(f"  Error loading dataset {name}: {e}")
            continue
    
    return train_datasets, eval_datasets

# Load datasets
train_datasets, eval_datasets = load_datasets_with_limits(config)

## Process Datasets for Training

Now let's process our datasets to prepare them for training.

In [ ]:
# Custom preprocessing function for text datasets
def preprocess_text_dataset(dataset, tokenizer, max_length=512, text_column="text"):
    """Preprocess dataset with tokenization for language model training."""
    
    # First, identify the text column if it's not the default "text"
    text_columns = [col for col in dataset.column_names if col.lower() in ['text', 'content', 'review', 'input', 'document']]
    if text_column not in dataset.column_names and text_columns:
        text_column = text_columns[0]
    
    # Handling special cases for known datasets
    if 'imdb' in dataset.info.builder_name.lower():
        text_column = 'text'
        if text_column not in dataset.column_names and 'review' in dataset.column_names:
            # Use review column for IMDB
            text_column = 'review'
    elif 'dialog' in dataset.info.builder_name.lower():
        # For dialog datasets, format conversations properly
        if 'conversations' in dataset.column_names:
            def format_dialog(example):
                conversation = ""
                for turn in example['conversations']:
                    role = turn.get('role', 'user')
                    content = turn.get('value', '')
                    conversation += f"{role}: {content}\n"
                return {"text": conversation}
            
            dataset = dataset.map(format_dialog)
            text_column = 'text'
    
    # Define tokenization function
    def tokenize_function(examples):
        # Get text from the appropriate column
        texts = examples[text_column]
        
        # Add EOS token if needed
        if tokenizer.eos_token:
            texts = [t + tokenizer.eos_token for t in texts]
            
        # Tokenize
        tokenized = tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        # For causal language modeling, input_ids are also labels
        tokenized["labels"] = tokenized["input_ids"].clone()
        
        return tokenized
    
    # Apply tokenization
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=[col for col in dataset.column_names if col != text_column]
    )
    
    return tokenized_dataset

# Get tokenizer
tokenizer = get_tokenizer(config['tokenizer'])
print(f"Loaded tokenizer with vocabulary size: {len(tokenizer)}")

# Process datasets
processed_train_datasets = [preprocess_text_dataset(dataset, tokenizer, 
                                                   max_length=config['model']['sizes'][config['model']['size']]['max_seq_length']) 
                           for dataset in train_datasets]

processed_eval_datasets = [preprocess_text_dataset(dataset, tokenizer, 
                                                  max_length=config['model']['sizes'][config['model']['size']]['max_seq_length']) 
                          for dataset in eval_datasets]

# Combine datasets if multiple are available
if len(processed_train_datasets) > 1:
    combined_train_dataset = concatenate_datasets(processed_train_datasets)
else:
    combined_train_dataset = processed_train_datasets[0] if processed_train_datasets else None
    
if len(processed_eval_datasets) > 1:
    combined_eval_dataset = concatenate_datasets(processed_eval_datasets)
else:
    combined_eval_dataset = processed_eval_datasets[0] if processed_eval_datasets else None

print(f"Processed {len(combined_train_dataset) if combined_train_dataset else 0} training examples")
print(f"Processed {len(combined_eval_dataset) if combined_eval_dataset else 0} evaluation examples")

## Create and Initialize Model

Now, let's create our model with the configuration.

In [ ]:
# Create model from scratch
print("Creating model with configuration...")
model = create_model_from_config(config)

# Print model size
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {total_params:,} parameters")

# Apply PEFT if enabled
if config['peft']['enabled']:
    peft_config = config['peft']
    peft_method = peft_config['method']
    print(f"Applying {peft_method.upper()} for parameter-efficient fine-tuning")
    
    # Prepare model for QLoRA if needed
    if peft_method == 'qlora' and BITSANDBYTES_AVAILABLE:
        qlora_config = peft_config['qlora']
        model = prepare_for_qlora(
            model,
            bits=qlora_config.get('bits', 4),
            groupsize=qlora_config.get('bnb_blocksize', 128)
        )
    
    # Get PEFT method config
    method_config = peft_config[peft_method]
    
    # Apply PEFT
    trainable_params_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    model = apply_peft_to_model(
        model=model,
        peft_config=method_config,
        peft_type=peft_method
    )
    
    trainable_params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters before PEFT: {trainable_params_before:,} ({trainable_params_before/total_params:.2%})")
    print(f"Trainable parameters after PEFT: {trainable_params_after:,} ({trainable_params_after/total_params:.2%})")
    print(f"Parameter reduction: {trainable_params_before/trainable_params_after:.1f}x")

## Train the Model

Now, let's train our model on the combined datasets.

In [ ]:
# Function to train the model
def train_model(model, train_dataset, eval_dataset, config):
    """Train model using our training pipeline."""
    # Get active stage
    active_stage = config['training']['active_stage']
    print(f"Training model for stage: {active_stage}")
    
    # Create training arguments
    training_args = TrainingArguments(config, active_stage)
    
    # Overrides for notebook environment
    training_args.epochs = 1  # Just one epoch for demonstration
    
    # Create trainer
    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args
    )
    
    # Train the model
    print("Starting training...")
    start_time = time.time()
    training_results = trainer.train()
    training_time = time.time() - start_time
    
    print(f"Training completed in {training_time:.2f} seconds")
    print(f"Training results: {training_results}")
    
    return trainer, training_results

# Train model if datasets are available
if combined_train_dataset is not None:
    trainer, results = train_model(model, combined_train_dataset, combined_eval_dataset, config)
else:
    print("No training datasets available. Skipping training.")

## Evaluate the Model

Let's evaluate our trained model.

In [ ]:
# Function to evaluate the model
def evaluate_trained_model(model, eval_dataset, tokenizer, config):
    """Evaluate the trained model."""
    print("Evaluating model...")
    # Set model to evaluation mode
    model.eval()
    
    # Get evaluation metrics
    metrics = evaluate_model(
        model=model,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        batch_size=config['data_processing']['batching']['eval_batch_size'],
        metrics=config['evaluation']['metrics'],
        output_dir=os.path.join(config['output_dir'], 'evaluation'),
    )
    
    print("Evaluation results:")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value:.4f}")
    
    return metrics

# Sample texts for generation testing
sample_texts = [
    "I really enjoyed this movie because",
    "The artificial intelligence revolution has",
    "In the near future, quantum computers will"
]

# Function to test generation
def test_generation(model, tokenizer, texts, config):
    """Test generation capabilities of the model."""
    print("Testing text generation capabilities...")
    model.eval()
    
    generation_config = config['evaluation']['generation']
    
    for i, text in enumerate(texts):
        print(f"\nSample {i+1}: {text}")
        
        # Tokenize input
        inputs = tokenizer(text, return_tensors="pt")
        input_ids = inputs.input_ids.to(model.device)
        
        # Generate text
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_length=len(input_ids[0]) + generation_config.get('max_length', 64),
                temperature=generation_config.get('temperature', 0.8),
                top_p=generation_config.get('top_p', 0.92),
                top_k=generation_config.get('top_k', 50),
                do_sample=generation_config.get('do_sample', True)
            )
        
        # Decode and display generated text
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Generated: {generated_text}")

# Evaluate and test model
if combined_eval_dataset is not None:
    metrics = evaluate_trained_model(model, combined_eval_dataset, tokenizer, config)
    test_generation(model, tokenizer, sample_texts, config)
else:
    print("No evaluation dataset available. Skipping evaluation.")

## Save and Export Model

Finally, let's save and export our trained model.

In [ ]:
# Function to save the trained model
def save_trained_model(model, tokenizer, config):
    """Save the trained model and tokenizer."""
    print("Saving trained model...")
    
    # Create final output directory
    final_output_dir = os.path.join(config['output_dir'], 'final_model')
    os.makedirs(final_output_dir, exist_ok=True)
    
    # Merge PEFT weights if using LoRA and merging is enabled
    if (config['peft']['enabled'] and config['peft']['method'] == 'lora' and 
        config['training']['checkpointing'].get('merge_lora_weights', False) and
        hasattr(model, 'merge_peft_weights')):
        print("Merging LoRA weights into base model...")
        model.merge_peft_weights()
        print("LoRA weights merged successfully")
    
    # Save the model
    model.save_pretrained(final_output_dir)
    tokenizer.save_pretrained(final_output_dir)
    
    # Save configuration
    with open(os.path.join(final_output_dir, 'training_config.yaml'), 'w') as f:
        yaml.dump(config, f)
    
    print(f"Model, tokenizer, and config saved to {final_output_dir}")
    
    return final_output_dir

# Function to test the saved model
def test_saved_model(model_path, text="The future of artificial intelligence is"):
    """Test loading and using the saved model."""
    print(f"\nTesting saved model from {model_path}...")
    
    try:
        # Load model and tokenizer
        loaded_model = AutoModelForCausalLM.from_pretrained(model_path)
        loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        # Ensure model is in evaluation mode
        loaded_model.eval()
        
        # Prepare input
        inputs = loaded_tokenizer(text, return_tensors="pt")
        
        # Generate text
        with torch.no_grad():
            outputs = loaded_model.generate(
                inputs.input_ids,
                max_length=len(inputs.input_ids[0]) + 50,
                temperature=0.8,
                do_sample=True,
                top_p=0.92,
            )
        
        # Decode and display generated text
        generated_text = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Input: {text}")
        print(f"Generated: {generated_text}")
        
        print("Saved model tested successfully!")
        return True
    except Exception as e:
        print(f"Error testing saved model: {e}")
        return False

# Save and test model
final_model_path = save_trained_model(model, tokenizer, config)
test_saved_model(final_model_path)

## Loading and Using the Model

Below is code for loading and using the model in your applications:

In [ ]:
# Example code for loading and using the trained model
print("Example code for loading and using the model:")
print("""
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the model and tokenizer
model_path = "./output_real_model/final_model"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set model to evaluation mode
model.eval()

# Generate text
def generate_text(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_length=max_length,
            temperature=0.8,
            top_p=0.92,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
text = generate_text("The future of AI is")
print(text)
""")

## Conclusion

Congratulations! You have successfully:

1. Loaded and processed real datasets with appropriate sample limits
2. Created and initialized a transformer model with modern architecture features
3. Applied parameter-efficient fine-tuning with LoRA
4. Trained the model on combined datasets
5. Evaluated the model and tested its generation capabilities
6. Saved and exported the model for production use

This notebook provides a complete end-to-end workflow that you can adapt for your specific datasets and requirements.